In [ ]:
# -*- coding: utf-8 -*-

import time
import random
import warnings
import msvcrt
from datetime import datetime

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, add_self_loops, to_networkx
from torch_geometric.nn import GCNConv

warnings.filterwarnings("ignore")


# ==========================
# 設定
# ==========================
TXS_FEATURES = "./transactions/txs_features.txt"
TXS_CLASSES  = "./transactions/txs_classes.txt"
TXS_EDGES    = "./transactions/txs_edgelist.txt"

SEED = 42

TRAIN_END_STEP = 34
VAL_END_STEP = 39

EPOCHS = 150
LR = 0.001
WEIGHT_DECAY = 5e-4
HIDDEN_DIM = 64
DROPOUT = 0.5

BETWEENNESS_K = 100

PSEUDO_LIST = [50, 75, 100]
PSEUDO_ONLY_TRAIN_PERIOD = True

INPUT_TIMEOUT_SEC = 10
INPUT_DEFAULT = "y"

OUTPUT_CENTER_ONLY = "unknown_outlier_center_only.csv"
OUTPUT_UNKNOWN_DETAIL_1ST = "unknown_outlier_with_gcn_score_first.csv"
OUTPUT_TIME_STEP_SUMMARY_1ST = "unknown_outlier_summary_by_time_step_first.csv"
OUTPUT_COMPARE_SUMMARY = "pseudo_illicit_compare_summary.csv"


# ==========================
# ログ
# ==========================
START_TIME = None

def log(msg):
    t = int(time.time() - START_TIME)
    print(f"[{datetime.now().strftime('%H:%M:%S')} +{t}s] {msg}")


def input_timeout(prompt, timeout=10, default="y"):
    print(
        f"{prompt} ※{timeout}秒入力がなければ [{default}] で続行します: ",
        end="",
        flush=True
    )

    start = time.time()
    chars = []

    while time.time() - start < timeout:
        if msvcrt.kbhit():
            ch = msvcrt.getwch()

            if ch in ("\r", "\n"):
                print()
                break

            chars.append(ch)
            print(ch, end="", flush=True)

        time.sleep(0.05)

    else:
        print()
        return default

    s = "".join(chars).strip()
    return s if s else default


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)


def robust_z(s):
    s = pd.to_numeric(s, errors="coerce").fillna(0)
    med = s.median()
    mad = (s - med).abs().median()

    if mad == 0:
        return (s - s.mean()) / (s.std() + 1e-9)

    return 0.6745 * (s - med) / mad


# ==========================
# GCN
# ==========================
class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim=2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)

        return self.lin(x)


def tune_threshold(y_true, prob_illicit):
    best_th = 0.5
    best_f1 = -1

    for th in np.arange(0.05, 0.96, 0.05):
        pred = (prob_illicit >= th).astype(int)
        f1 = f1_score(y_true, pred, pos_label=1, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    return best_th, best_f1


def extract_metrics(report_dict):
    return {
        "illicit_precision": report_dict["illicit"]["precision"],
        "illicit_recall": report_dict["illicit"]["recall"],
        "illicit_f1": report_dict["illicit"]["f1-score"],
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    }


def run_gcn_training(
    data_base,
    X_raw,
    y_train_np,
    original_y_np,
    time_steps_np,
    device,
    label_name="first"
):
    log(f"GCN学習開始: {label_name}")

    train_mask_np = (
        (y_train_np != -1) &
        (time_steps_np <= TRAIN_END_STEP)
    )

    val_mask_np = (
        (original_y_np != -1) &
        (time_steps_np > TRAIN_END_STEP) &
        (time_steps_np <= VAL_END_STEP)
    )

    test_mask_np = (
        (original_y_np != -1) &
        (time_steps_np > VAL_END_STEP)
    )

    log(
        f"{label_name}: train={train_mask_np.sum()} "
        f"val={val_mask_np.sum()} "
        f"test={test_mask_np.sum()}"
    )

    scaler = StandardScaler()
    X_scaled = X_raw.astype(np.float32).copy()

    X_scaled[train_mask_np] = scaler.fit_transform(X_scaled[train_mask_np])
    X_scaled[~train_mask_np] = scaler.transform(X_scaled[~train_mask_np])

    data = Data(
        x=torch.tensor(X_scaled, dtype=torch.float),
        edge_index=data_base.edge_index.clone(),
        y=torch.tensor(y_train_np, dtype=torch.long),
        time_steps=data_base.time_steps.clone(),
        node_ids=data_base.node_ids.clone()
    )

    data.train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
    data.val_mask = torch.tensor(val_mask_np, dtype=torch.bool)
    data.test_mask = torch.tensor(test_mask_np, dtype=torch.bool)

    data = data.to(device)

    model = GCN(
        in_dim=data.x.size(1),
        hidden_dim=HIDDEN_DIM,
        out_dim=2
    ).to(device)

    train_y = data.y[data.train_mask]
    counts = torch.bincount(train_y, minlength=2).float()

    weights = counts.sum() / (counts + 1e-9)
    weights = weights / weights.mean()
    weights = weights.to(device)

    log(f"{label_name}: class counts={counts.detach().cpu().numpy()}")
    log(f"{label_name}: class weights={weights.detach().cpu().numpy()}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    best_val_f1 = -1
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)

        loss = F.cross_entropy(
            out[data.train_mask],
            data.y[data.train_mask],
            weight=weights
        )

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            prob = F.softmax(out, dim=1)

            val_prob_illicit = prob[data.val_mask, 0].detach().cpu().numpy()
            val_true_raw = original_y_np[val_mask_np]
            val_true = (val_true_raw == 0).astype(int)

            th, val_f1 = tune_threshold(val_true, val_prob_illicit)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {
                "model": model.state_dict(),
                "threshold": th,
                "epoch": epoch
            }

        if epoch == 1 or epoch % 10 == 0:
            log(
                f"{label_name}: epoch={epoch:03d} "
                f"loss={loss.item():.6f} "
                f"val_illicit_f1={val_f1:.4f} "
                f"best={best_val_f1:.4f} "
                f"th={th:.2f}"
            )

    if best_state is None:
        raise RuntimeError("best_state が作成されませんでした")

    model.load_state_dict(best_state["model"])
    model.eval()

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        prob = F.softmax(out, dim=1)

    test_prob_illicit = prob[data.test_mask, 0].detach().cpu().numpy()
    test_true_raw = original_y_np[test_mask_np]
    test_true = (test_true_raw == 0).astype(int)
    test_pred = (test_prob_illicit >= best_state["threshold"]).astype(int)

    cm = confusion_matrix(test_true, test_pred)

    report_text = classification_report(
        test_true,
        test_pred,
        target_names=["licit", "illicit"],
        zero_division=0
    )

    report_dict = classification_report(
        test_true,
        test_pred,
        target_names=["licit", "illicit"],
        zero_division=0,
        output_dict=True
    )

    log(f"GCN学習完了: {label_name}")

    return {
        "prob": prob.detach().cpu().numpy(),
        "best_epoch": best_state["epoch"],
        "best_threshold": best_state["threshold"],
        "best_val_f1": best_val_f1,
        "test_cm": cm,
        "test_report_text": report_text,
        "test_report_dict": report_dict,
        "train_count": int(train_mask_np.sum()),
        "val_count": int(val_mask_np.sum()),
        "test_count": int(test_mask_np.sum())
    }


def add_gcn_and_combined_score(df_u, prob_all, suffix=""):
    df = df_u.copy()

    prob_illicit_all = prob_all[:, 0]

    col_prob = f"gcn_illicit_prob{suffix}"
    col_combined = f"combined_score{suffix}"

    df[col_prob] = df["node_index"].map(
        lambda idx: float(prob_illicit_all[int(idx)])
    )

    df[col_combined] = (
        df["centrality_score"].rank(pct=True) * 0.5
        + df[col_prob].rank(pct=True) * 0.5
    )

    return df


def make_summary(df, prob_col, score_col):
    return df.groupby("time_step").agg(
        unknown_count=(score_col, "count"),
        avg_degree=("degree", "mean"),
        avg_betweenness=("betweenness", "mean"),
        avg_pagerank=("pagerank", "mean"),
        max_centrality_score=("centrality_score", "max"),
        avg_centrality_score=("centrality_score", "mean"),
        avg_gcn_illicit_prob=(prob_col, "mean"),
        max_gcn_illicit_prob=(prob_col, "max"),
        avg_combined_score=(score_col, "mean"),
        max_combined_score=(score_col, "max")
    ).reset_index()


def main():
    global START_TIME
    START_TIME = time.time()

    log("START")
    set_seed()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"device={device}")

    log("CSV読み込み開始")

    df_f = pd.read_csv(TXS_FEATURES)
    df_c = pd.read_csv(TXS_CLASSES)
    df_e = pd.read_csv(TXS_EDGES)

    log(f"features={df_f.shape} classes={df_c.shape} edges={df_e.shape}")

    id_col_f = df_f.columns[0]
    id_col_c = df_c.columns[0]

    df = df_f.merge(df_c, left_on=id_col_f, right_on=id_col_c, how="left")

    if df["class"].isna().any():
        log("警告: class欠損があります。unknown扱いにします")
        df["class"] = df["class"].fillna(3)

    ids = df[id_col_f].astype(int).values
    id2idx = {int(v): i for i, v in enumerate(ids)}

    y_np = df["class"].map(
        lambda x: 0 if int(x) == 1 else 1 if int(x) == 2 else -1
    ).values

    original_y_np = y_np.copy()
    time_steps_np = df["Time step"].astype(int).values

    log(
        f"illicit={(y_np == 0).sum()} "
        f"licit={(y_np == 1).sum()} "
        f"unknown={(y_np == -1).sum()}"
    )

    exclude_cols = {
        id_col_f,
        id_col_c,
        "class",
        "Time step"
    }

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    log(f"特徴量数={len(feature_cols)}")

    X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_raw = X.values.astype(np.float32)

    log("edge構築開始")

    col0, col1 = df_e.columns[:2]

    edges = []
    skipped = 0

    for u, v in zip(df_e[col0], df_e[col1]):
        try:
            u = int(u)
            v = int(v)
        except Exception:
            skipped += 1
            continue

        if u in id2idx and v in id2idx:
            edges.append([id2idx[u], id2idx[v]])
        else:
            skipped += 1

    log(f"edge構築完了: edges={len(edges)} skipped={skipped}")

    edge_index = torch.tensor(edges, dtype=torch.long).T
    edge_index = to_undirected(edge_index)
    edge_index, _ = add_self_loops(edge_index, num_nodes=len(ids))

    data_base = Data(
        x=torch.tensor(X_raw, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y_np, dtype=torch.long),
        time_steps=torch.tensor(time_steps_np, dtype=torch.long),
        node_ids=torch.tensor(ids, dtype=torch.long)
    )

    log("中心性計算開始")

    G = to_networkx(data_base, to_undirected=True)
    G.remove_edges_from(nx.selfloop_edges(G))

    log(f"Graph nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

    log("degree計算")
    deg = dict(G.degree())

    log("betweenness計算開始")
    bet = nx.betweenness_centrality(G, k=BETWEENNESS_K, seed=SEED)

    log("pagerank計算開始")
    pr = nx.pagerank(G)

    log("中心性計算完了")

    rows = []

    for i in range(len(y_np)):
        if y_np[i] != -1:
            continue

        rows.append({
            "txId": int(ids[i]),
            "node_index": int(i),
            "time_step": int(time_steps_np[i]),
            "degree": deg.get(i, 0),
            "betweenness": bet.get(i, 0),
            "pagerank": pr.get(i, 0)
        })

    df_u = pd.DataFrame(rows)
    log(f"unknown数={len(df_u)}")

    df_u["d_z"] = robust_z(df_u["degree"])
    df_u["b_z"] = robust_z(df_u["betweenness"])
    df_u["p_z"] = robust_z(df_u["pagerank"])

    df_u["centrality_score"] = df_u[["d_z", "b_z", "p_z"]].abs().max(axis=1)

    def reason(row):
        vals = {
            "degree": abs(row["d_z"]),
            "betweenness": abs(row["b_z"]),
            "pagerank": abs(row["p_z"])
        }
        k = max(vals, key=vals.get)

        if k == "degree":
            return "大量接続ノード"
        elif k == "betweenness":
            return "資金流路の橋渡しノード"
        else:
            return "重要ノード群に接続する目立つノード"

    df_u["centrality_reason"] = df_u.apply(reason, axis=1)

    df_u.to_csv(OUTPUT_CENTER_ONLY, index=False, encoding="utf-8-sig")
    log(f"CSV保存: {OUTPUT_CENTER_ONLY}")

    ans = input_timeout(
        "\nGCN学習を実行しますか？ [y/N]",
        timeout=INPUT_TIMEOUT_SEC,
        default=INPUT_DEFAULT
    ).strip().lower()

    if ans not in ["y", "yes"]:
        log("GCN学習をスキップします")
        log("DONE")
        return

    # ======================
    # 1回目
    # ======================
    result_first = run_gcn_training(
        data_base=data_base,
        X_raw=X_raw,
        y_train_np=y_np,
        original_y_np=original_y_np,
        time_steps_np=time_steps_np,
        device=device,
        label_name="first"
    )

    df_first = add_gcn_and_combined_score(
        df_u=df_u,
        prob_all=result_first["prob"],
        suffix="_first"
    )

    summary_first = make_summary(
        df_first,
        prob_col="gcn_illicit_prob_first",
        score_col="combined_score_first"
    )

    df_first.to_csv(
        OUTPUT_UNKNOWN_DETAIL_1ST,
        index=False,
        encoding="utf-8-sig"
    )

    summary_first.to_csv(
        OUTPUT_TIME_STEP_SUMMARY_1ST,
        index=False,
        encoding="utf-8-sig"
    )

    log(f"CSV保存: {OUTPUT_UNKNOWN_DETAIL_1ST}")
    log(f"CSV保存: {OUTPUT_TIME_STEP_SUMMARY_1ST}")

    # ======================
    # Top50 / 75 / 100
    # ======================
    compare_rows = []
    detailed_results = {}

    for top_n in PSEUDO_LIST:
        log(f"==== pseudo illicit Top{top_n} 開始 ====")

        df_ranked = df_first.sort_values(
            "combined_score_first",
            ascending=False
        ).copy()

        if PSEUDO_ONLY_TRAIN_PERIOD:
            df_ranked = df_ranked[df_ranked["time_step"] <= TRAIN_END_STEP]
            log(f"Top{top_n}: train期間内 unknown のみに制限")

        df_pseudo = df_ranked.head(top_n).copy()
        pseudo_indices = df_pseudo["node_index"].astype(int).values

        log(f"Top{top_n}: pseudo illicit 数={len(pseudo_indices)}")

        df_pseudo["pseudo_label"] = "illicit"
        df_pseudo["pseudo_label_value"] = 0
        df_pseudo["pseudo_source"] = f"combined_rank_top{top_n}_first"

        pseudo_file = f"pseudo_illicit_combined_rank_top{top_n}.csv"
        df_pseudo.to_csv(pseudo_file, index=False, encoding="utf-8-sig")
        log(f"CSV保存: {pseudo_file}")

        y_np_recalc = y_np.copy()
        y_np_recalc[pseudo_indices] = 0

        result = run_gcn_training(
            data_base=data_base,
            X_raw=X_raw,
            y_train_np=y_np_recalc,
            original_y_np=original_y_np,
            time_steps_np=time_steps_np,
            device=device,
            label_name=f"top{top_n}"
        )

        df_recalc = add_gcn_and_combined_score(
            df_u=df_u,
            prob_all=result["prob"],
            suffix=f"_top{top_n}"
        )

        df_recalc["gcn_illicit_prob_first"] = df_first["gcn_illicit_prob_first"]
        df_recalc["combined_score_first"] = df_first["combined_score_first"]
        df_recalc["pseudo_illicit_used"] = df_recalc["node_index"].isin(pseudo_indices)

        detail_file = f"unknown_outlier_with_gcn_score_top{top_n}.csv"
        summary_file = f"unknown_outlier_summary_by_time_step_top{top_n}.csv"

        df_recalc.to_csv(detail_file, index=False, encoding="utf-8-sig")

        summary_recalc = make_summary(
            df_recalc,
            prob_col=f"gcn_illicit_prob_top{top_n}",
            score_col=f"combined_score_top{top_n}"
        )

        summary_recalc.to_csv(summary_file, index=False, encoding="utf-8-sig")

        log(f"CSV保存: {detail_file}")
        log(f"CSV保存: {summary_file}")

        metrics = extract_metrics(result["test_report_dict"])
        cm = result["test_cm"]

        compare_rows.append({
            "mode": f"pseudo_top{top_n}",
            "pseudo_top_n": top_n,
            "pseudo_count": len(pseudo_indices),
            "train_count": result["train_count"],
            "val_count": result["val_count"],
            "test_count": result["test_count"],
            "best_epoch": result["best_epoch"],
            "best_threshold": result["best_threshold"],
            "best_val_f1": result["best_val_f1"],
            "tn_licit_correct": int(cm[0, 0]),
            "fp_licit_to_illicit": int(cm[0, 1]),
            "fn_illicit_to_licit": int(cm[1, 0]),
            "tp_illicit_correct": int(cm[1, 1]),
            **metrics
        })

        detailed_results[top_n] = result

        log(f"==== pseudo illicit Top{top_n} 完了 ====")

    # ======================
    # 1回目も比較に追加
    # ======================
    cm_first = result_first["test_cm"]
    metrics_first = extract_metrics(result_first["test_report_dict"])

    first_row = {
        "mode": "first_no_pseudo",
        "pseudo_top_n": 0,
        "pseudo_count": 0,
        "train_count": result_first["train_count"],
        "val_count": result_first["val_count"],
        "test_count": result_first["test_count"],
        "best_epoch": result_first["best_epoch"],
        "best_threshold": result_first["best_threshold"],
        "best_val_f1": result_first["best_val_f1"],
        "tn_licit_correct": int(cm_first[0, 0]),
        "fp_licit_to_illicit": int(cm_first[0, 1]),
        "fn_illicit_to_licit": int(cm_first[1, 0]),
        "tp_illicit_correct": int(cm_first[1, 1]),
        **metrics_first
    }

    df_compare = pd.DataFrame([first_row] + compare_rows)

    df_compare.to_csv(
        OUTPUT_COMPARE_SUMMARY,
        index=False,
        encoding="utf-8-sig"
    )

    log(f"CSV保存: {OUTPUT_COMPARE_SUMMARY}")

    # ======================
    # 詳細レポート
    # ======================
    print("\n==============================")
    print("      GCN 学習結果まとめ 1回目")
    print("==============================")
    print(f"train_count       : {result_first['train_count']}")
    print(f"val_count         : {result_first['val_count']}")
    print(f"test_count        : {result_first['test_count']}")
    print(f"best_epoch        : {result_first['best_epoch']}")
    print(f"best_threshold    : {result_first['best_threshold']:.2f}")
    print(f"best_val_f1       : {result_first['best_val_f1']:.6f}")
    print("\n=== Test Confusion Matrix 1回目 ===")
    print(result_first["test_cm"])
    print("\n=== Test Classification Report 1回目 ===")
    print(result_first["test_report_text"])

    for top_n in PSEUDO_LIST:
        result = detailed_results[top_n]

        print("\n==============================")
        print(f"      GCN 学習結果まとめ Top{top_n}")
        print("==============================")
        print(f"pseudo_illicit追加数 : {top_n}")
        print(f"train_count          : {result['train_count']}")
        print(f"val_count            : {result['val_count']}")
        print(f"test_count           : {result['test_count']}")
        print(f"best_epoch           : {result['best_epoch']}")
        print(f"best_threshold       : {result['best_threshold']:.2f}")
        print(f"best_val_f1          : {result['best_val_f1']:.6f}")
        print(f"\n=== Test Confusion Matrix Top{top_n} ===")
        print(result["test_cm"])
        print(f"\n=== Test Classification Report Top{top_n} ===")
        print(result["test_report_text"])

    # ======================
    # 最後に指定形式の比較表を表示
    # ======================
    print("\n==============================")
    print("        最終比較表")
    print("==============================")

    df_final = df_compare.copy()

    order = ["first_no_pseudo", "pseudo_top50", "pseudo_top75", "pseudo_top100"]
    df_final["mode"] = pd.Categorical(
        df_final["mode"],
        categories=order,
        ordered=True
    )
    df_final = df_final.sort_values("mode")

    df_final_show = pd.DataFrame({
        "条件": [
            "1回目" if m == "first_no_pseudo"
            else f"Top{int(n)}"
            for m, n in zip(df_final["mode"], df_final["pseudo_top_n"])
        ],
        "pseudo数": df_final["pseudo_count"],
        "閾値": df_final["best_threshold"],
        "検証F1": df_final["best_val_f1"],
        "適合率(P)": df_final["illicit_precision"],
        "再現率(R)": df_final["illicit_recall"],
        "F1": df_final["illicit_f1"],
        "正解率": df_final["accuracy"],
        "誤検出(FP)": df_final["fp_licit_to_illicit"],
        "見逃し(FN)": df_final["fn_illicit_to_licit"],
        "検出(TP)": df_final["tp_illicit_correct"]
    })

    print(
        df_final_show.to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}"
        )
    )

    print("\n※ 誤検出(FP) = licit を illicit と誤判定")
    print("※ 見逃し(FN) = illicit を licit と誤判定")
    print("※ 検出(TP) = illicit を正しく illicit と判定")

    # ======================
    # 最良モデル
    # ======================
    best_row = df_compare.sort_values("illicit_f1", ascending=False).iloc[0]

    print("\n==============================")
    print("        最良モデル F1基準")
    print("==============================")
    print(f"条件       : {best_row['mode']}")
    print(f"pseudo数   : {best_row['pseudo_count']}")
    print(f"illicit F1 : {best_row['illicit_f1']:.4f}")
    print(f"再現率     : {best_row['illicit_recall']:.4f}")
    print(f"適合率     : {best_row['illicit_precision']:.4f}")
    print(f"閾値       : {best_row['best_threshold']:.2f}")

    log("DONE")


if __name__ == "__main__":
    main()